# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tal3at-M/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
#Editorial Prioritization Queue:
#- Pages are ranked by predicted decay probability generated by the trained ensemble model.
#- Each URL is mapped to an interpretable operational reason code:
 # - `EXPAND_CONTENT`: High impression pool with decaying clicks; needs updated subheadings, fresh statistics, and deeper coverage.
  #- `METADATA_REFRESH`: Solid average ranking (top 15) but suppressed CTR; requires title tag and meta description overhaul.
  #- `PRUNE_OR_REDIRECT`: Very low impressions, poor rank (>50), and prolonged age (>400 days); candidate for consolidation or 301 redirect.

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

url = "https://raw.githubusercontent.com/Tal3at-M/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Feature setup
df["ctr_calc"] = df["clicks_90d"] / (df["impressions_90d"] + 1e-5)
df["log_impressions"] = np.log1p(df["impressions_90d"])
df["log_clicks"] = np.log1p(df["clicks_90d"])

features = [
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "content_age_days",
    "log_impressions",
    "log_clicks",
    "ctr_calc",
]
X = df[features].fillna(df[features].median())
y = (df["trend_direction"].str.lower() == "down").astype(int)

# Train ranking model
rf = RandomForestClassifier(
    n_estimators=100, max_depth=6, random_state=42, n_jobs=-1
)
rf.fit(X, y)
df["decay_prob"] = rf.predict_proba(X)[:, 1]


def assign_action(row):
  if row["decay_prob"] >= 0.70 and row["impressions_90d"] >= 1000:
    return "EXPAND_CONTENT"
  elif row["avg_position"] <= 15.0 and row["ctr_calc"] < 0.01:
    return "METADATA_REFRESH"
  elif row["impressions_90d"] < 100 and row["content_age_days"] > 400:
    return "PRUNE_OR_REDIRECT"
  return "MONITOR"


df["suggested_action"] = df.apply(assign_action, axis=1)
ranked_queue = df.sort_values(by="decay_prob", ascending=False).reset_index(
    drop=True
)
ranked_queue["priority_rank"] = ranked_queue.index + 1

print(
    f"Ranked queue generated successfully: {len(ranked_queue):,} URLs analyzed."
)
ranked_queue[[
    "priority_rank",
    "content_id",
    "decay_prob",
    "suggested_action",
    "impressions_90d",
    "avg_position",
]].head(5)

Ranked queue generated successfully: 30,000 URLs analyzed.


,priority_rank,content_id,decay_prob,suggested_action,impressions_90d,avg_position
0,1,content_d225ec9f3d46,0.761465,EXPAND_CONTENT,26470,0.7
1,2,content_0be51c9e6cbd,0.759105,EXPAND_CONTENT,4205,0.7
2,3,content_6b09c696507d,0.754018,EXPAND_CONTENT,2813,0.8
3,4,content_67095eb7f5de,0.753030,EXPAND_CONTENT,4782,0.6
4,5,content_2a57f2016a14,0.751509,EXPAND_CONTENT,2324,0.6


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [8]:
#Intended Users:
#- Content Strategists and SEO Editors planning bi-weekly content refresh sprints.

#Scope and Operational Limits:
#- In-Scope: Identifies empirical visibility drops across organic Google search results.
#- Out-of-Scope: Does not account for direct business conversion value, paid ad cannibalization, seasonal query fluctuations (e.g., Black Friday spikes), or sitewide technical migrations.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [9]:
#Mandatory Human Verification:
#- Search Intent Audit: Editors must verify whether the original user search intent evolved before modifying copy.
#- Brand & Legal Compliance: Ensure suggested refreshes do not alter compliance-sensitive statements or product claims.

#The No-Go List (Never Automate):
#- Automatic Page Deletion / 404ing: High-traffic decaying URLs must never be pruned without direct editorial review.
#- Automated AI Rewriting: Complete autonomous text overwrites without human editing are prohibited to maintain brand voice and factual accuracy.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [10]:
#Production Monitoring & Retraining Signals:
#- Metric Drift: Retrain when out-of-sample Precision@50 drops below 65% across 2 consecutive sprints.
#- Distributional Drift: Retrain if median impressions or SERP rank distributions shift significantly post-Google core algorithm updates.
#- Recency Limit: Mandatory monthly retraining cadence using the latest rolling 90-day performance window.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [11]:
import os

os.makedirs("work/outputs", exist_ok=True)

# Export top 100 queue for paper appendices
export_cols = [
    "priority_rank",
    "content_id",
    "decay_prob",
    "suggested_action",
    "impressions_90d",
    "clicks_90d",
    "avg_position",
    "content_age_days",
]
output_file = "work/outputs/action_playbook_queue.csv"
ranked_queue[export_cols].head(100).to_csv(output_file, index=False)

print(f"Playbook export written to: {output_file}")
print(
    f"Action distribution across top 100 queue:\n{ranked_queue['suggested_action'].head(100).value_counts()}"
)

Playbook export written to: work/outputs/action_playbook_queue.csv
Action distribution across top 100 queue:
suggested_action
EXPAND_CONTENT      98
METADATA_REFRESH     2
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.